# Hazard indicators
Physrisk is primarily designed to run 'bottom-up' calculations that model the impact of climate hazards on large numbers of individual assets (including natural) and operations. These calculations can be used to assess financial risks or socio-economic impacts. To do this physrisk collects:

- hazard indicators and
- models of vulnerability of assets/operations to hazards.

Hazard indicators – that is, quantities that provide the information about hazards needed by the vulnerability models – are collected from a variety of sources. OS-Climate consolidates public domain hazard indicators and also provides the means to combine these with commercial data.

We start with public domain indicators: which ones are available and how can these be obtained?

## Requesting hazard indicators via the physrisk API

Using physrisk, hazard indicators can be obtained directly from source, as long as the user has the necessary API keys. Alternatively, the API – which is simply a hosted instance of physrisk – can be used, albeit the number of requests will be restricted to 30,000 latitudes and longitudes.

We give a walk-through of the hazard indicator API below, as well as examples of how to obtain the inventory of available hazard indicators.

In [ ]:
# pip install nbformat pandas plotly requests

In [ ]:
import pprint as pp
from IPython.display import Markdown, display
import pandas as pd
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import plotly.io
import requests

plotly.io.renderers.default = "notebook"

In [ ]:
import requests

base_url = "https://physrisk.com/api/"

request = {
    "items": [
        {
            "longitudes": [
                69.4787,
                68.71,
                20.1047,
                19.8936,
                19.6359,
                0.5407,
                6.9366,
                6.935,
                13.7319,
                13.7319,
            ],
            "latitudes": [
                34.556,
                35.9416,
                39.9116,
                41.6796,
                42.0137,
                35.7835,
                36.8789,
                36.88,
                -12.4706,
                -12.4706,
            ],
            "request_item_id": "my_flood_request",
            "hazard_type": "RiverineInundation",
            "indicator_id": "flood_depth",
            "scenario": "historical",
            "year": 1980,
        },
        {
            "longitudes": [
                69.4787,
                68.71,
                20.1047,
                19.8936,
                19.6359,
                0.5407,
                6.9366,
                6.935,
                13.7319,
                13.7319,
            ],
            "latitudes": [
                34.556,
                35.9416,
                39.9116,
                41.6796,
                42.0137,
                35.7835,
                36.8789,
                36.88,
                -12.4706,
                -12.4706,
            ],
            "request_item_id": "my_flood_request",
            "hazard_type": "RiverineInundation",
            "indicator_id": "flood_depth",
            "scenario": "rcp8p5",
            "indicator_model_gcm": "NorESM1-M",  # optional: can specify
            "year": 2050,
        },
        {
            "longitudes": [114.089],
            "latitudes": [22.4781],
            "request_item_id": "my_wind_request_ssp585",
            "hazard_type": "Wind",
            "indicator_id": "max_speed",
            "scenario": "historical",
            "path": "wind/iris/v1/max_speed_{scenario}_{year}",
            # if path is specified then that particular data array is used
            "year": 2010,
        },
        {
            "longitudes": [114.089],
            "latitudes": [22.4781],
            "request_item_id": "my_wind_request_histo",
            "hazard_type": "Wind",
            "indicator_id": "max_speed",
            "scenario": "ssp585",
            "path": "wind/iris/v1/max_speed_{scenario}_{year}",
            "year": 2050,
        },
        {
            "longitudes": [114.089],
            "latitudes": [22.4781],
            "request_item_id": "my_fire_request",
            "hazard_type": "Fire",
            "indicator_id": "fire_probability",
            "scenario": "ssp585",
            "path": "fire/jupiter/v1/fire_probability_{scenario}_{year}",
            "year": 2040,
        },
    ]
}
url = base_url + "get_hazard_data"
response = requests.post(url, json=request).json()
flood_results_baseline, flood_results_rcp585 = (
    response["items"][0]["intensity_curve_set"],
    response["items"][1]["intensity_curve_set"],
)
wind_results_baseline, wind_results_ssp585 = (
    response["items"][2]["intensity_curve_set"],
    response["items"][3]["intensity_curve_set"],
)
fire_results = response["items"][4]["intensity_curve_set"]

In [ ]:
from plotly.subplots import make_subplots
from IPython.display import HTML

fig1 = make_subplots(rows=1, cols=2)
fig1.add_scatter(
    x=flood_results_baseline[0]["index_values"],
    y=flood_results_baseline[0]["intensities"],
    name="baseline flood",
    row=1,
    col=1,
)
fig1.add_scatter(
    x=flood_results_rcp585[0]["index_values"],
    y=flood_results_rcp585[0]["intensities"],
    name="flood RCP 8.5 2050",
    row=1,
    col=1,
)
fig1.update_xaxes(
    title="Return period (years)", title_font={"size": 14}, row=1, col=1, type="log"
)
fig1.update_yaxes(title="Flood depth (m)", title_font={"size": 14}, row=1, col=1)
fig1.add_scatter(
    x=wind_results_baseline[0]["index_values"],
    y=wind_results_baseline[0]["intensities"],
    name="baseline wind",
    row=1,
    col=2,
)
fig1.add_scatter(
    x=wind_results_ssp585[0]["index_values"],
    y=wind_results_ssp585[0]["intensities"],
    name="wind SSP585 2050",
    row=1,
    col=2,
)
fig1.update_xaxes(
    title="Return period (years)", title_font={"size": 14}, row=1, col=2, type="log"
)
fig1.update_yaxes(
    title="Max (1 minute) wind speed (m/s)", title_font={"size": 14}, row=1, col=2
)
fig1.update_layout(legend=dict(orientation="h", y=-0.15))
fig1.update_layout(margin=dict(l=20, r=20, t=20, b=20))

# Render explicitly with include_mathjax=False: the default "notebook" renderer
# always embeds its own (legacy v2) MathJax, which conflicts with the MathJax
# already loaded by Sphinx for the $...$ math elsewhere on this page.
HTML(
    fig1.to_html(
        div_id="hazard-indicators-flood-wind",
        include_plotlyjs="cdn",
        include_mathjax=False,
        full_html=False,
    )
)

#### Requesting hazard indicators through physrisk directly

As mentioned above, the API is simply using a hosted version of physrisk and it is possible to run the same calculations using physrisk directly, as long as the necessary API keys are present in a credentials.env file.

The requests above can be run in physrisk using a Requester object.

In [ ]:
# pip install physrisk-lib

In [ ]:
from physrisk.container import Container

# the container is a dependency injection container,
# which allows the calculation to be configured to a particular use-case
container = Container()
# the requester is used to run calculations using the API.
# At this point, we can of course debug into the code and modify as required.
requester = container.requester()
result = requester.get(request_id="get_hazard_data", request_dict=request)